##### 소규모 모델 PEFT(LoRA)튜닝
- 가상의 문장을 생성
- train.jsonl, validation.jsonl 생성
- Transformers + PEFT(LoRA) fineTurning

In [12]:
import torch
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
train_file = r'finetune_data\train.jsonl'
valid_file = r'finetune_data\validation.jsonl'
output_dir = 'peft_output'
num_train_epochs = 3
per_device_train_batch_size = 4
learning_rate = 2e-4
# gpu
if torch.cuda.is_available():    
    use_bf16 = True
    use_fp16= False    
# cpu
else:
    use_bf16 = False
    use_fp16= True

lora_r = 8
lora_alpha = 16
lora_dropout = 0.05
use_4bit = False   
max_leng = 512

LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}
print('Config set:', model_name)

Config set: Qwen/Qwen2.5-0.5B-Instruct


In [13]:
# Causal lM : 기본학습 다음토큰을 예측
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def build_inputs_and_labels(batch, tokenizer, max_length=512):
    '''prompt부분은 loss에서제외(-100) 정답 라벨 토큰만 학습'''
    inputs = []; labels = []
    for p, r in zip(batch['prompt'], batch['response']):
        full = p+' '+ r
        tokenized_full = tokenizer(full, truncation=True, max_length=max_length)
        tokenized_prompt = tokenizer(p, truncation=True, max_length=max_length)
        input_ids =  tokenized_full['input_ids']
        label_ids = input_ids.copy()

        prompt_len = len(tokenized_prompt['input_ids'])
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i]    = -100         
        inputs.append(input_ids)
        labels.append(label_ids)
    return {'input_ids' : inputs, 'labels':labels}

In [14]:
# 토크나이져, 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
load_kwargs = {'trust_remote_code':True}
if use_4bit:
    load_kwargs.update({'load_in_4bit':True,'device_map':'auto'})
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
if use_4bit:
    model = prepare_model_for_kbit_training(model)
print('model load')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model load


##### LoRA 구성 및 적용
- 모델내부 선형층 이름을 스캔해 LoRA를 걸 target_modules를 자동으로 찾기
- attention/MLP projection에 LoRA를 적용?
    - 출력헤드 전체를 모두 fine tune하면 비용증가
    - 출력에 영향이 큰 선형층을 효율적으로 조정
- LoRA는 기존 가중치 W를 직접 바꾸지 않고 행렬 업데이트를통해서 조금 수정
- 학습 파라메터가 줄어서 cpu나 저성능의 gpu에서 가능

In [15]:
import torch.nn as nn
def find_lora_target_modules(model):
    linear_leaf_names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            linear_leaf_names.add(name.split('.')[-1])

    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    selected = [ m for m in preferred if m in linear_leaf_names]
    if not selected:
        fallback_keywords = ('proj','wq','wk','wv','wo','fc')
        selected = sorted([
            n for n in linear_leaf_names
            if any( k in  n for k in fallback_keywords) and n not in {'lm_head'}
        ])
    if not selected:
        raise ValueError(
            f'LoRA Target Module를 자동 탐색하지 못했습니다. 발견된 leaner leaf name : {linear_leaf_names}'
        )
    return selected, sorted(linear_leaf_names)

target_modules, linear_names = find_lora_target_modules(model)
print('Detected linear left names(samples) :',linear_names[:30])
print('Selected target_modules for LoRA', target_modules)

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Detected linear left names(samples) : ['down_proj', 'gate_proj', 'k_proj', 'lm_head', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Selected target_modules for LoRA ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


#### 데이터셋 로드 및 전처리


In [16]:
data_files = {'train':train_file, 'validation':valid_file}
dataset = load_dataset('json',data_files=data_files)

def preprocess_batch(batch):
    return build_inputs_and_labels(batch, tokenizer,max_length=max_leng)

tokenized_train = dataset['train'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['train'].column_names
)
tokenized_eval = dataset['validation'].map(
    lambda x : preprocess_batch(x), batched=True,remove_columns=dataset['validation'].column_names
)

def collate_with_padding(features):
    # 가변 길이 샘플을 동적 패딩하고 labels는 -100으로 패딩
    input_ids = [f['input_ids'] for f in features]
    labels = [f['labels'] for f in features]

    padded = tokenizer.pad({'input_ids': input_ids}, padding=True, return_tensors='pt')
    max_len = padded['input_ids'].shape[1]

    padded_labels = []
    for lb in labels:
        padded_labels.append(lb + [-100] * (max_len - len(lb)))

    padded['labels'] = torch.tensor(padded_labels, dtype=torch.long)
    return padded


print('Datasets prepared:', len(tokenized_train), len(tokenized_eval))

Datasets prepared: 2000 400


##### Trainer 구성 및 학습시작

In [ ]:
from transformers import TrainingArguments, Trainer
base_kwargs = dict(
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    learning_rate=learning_rate,
    bf16=use_bf16,
    fp16=use_fp16,
    save_total_limit=3,
    remove_unused_columns=False,
    logging_steps=10,
)

try:
    training_args =  TrainingArguments(**{**base_kwargs, 'evaluation_strategy':'epoch'})
except TypeError:
    training_args =  TrainingArguments(**base_kwargs)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator = collate_with_padding
)
train_result = trainer.train()
print(f"loss : {getattr(train_result,'training_loss', None)}")

c:\Users\Playdata\miniconda3\envs\torch_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
